# EuroSAT 数据集物理划分脚本

**目标**:
该脚本将根据之前 `data_preprocessing.ipynb` 生成的 `train.csv`, `valid.csv`, `test.csv` 文件，将原始图片**复制**到一个新的、按 `train/valid/test` 结构组织的文件夹中。

**核心功能**:
1.  在项目根目录下创建一个名为 `EuroSAT_Split_Physical` 的新文件夹。
2.  在这个新文件夹内，创建 `train`, `valid`, 和 `test` 三个子文件夹。
3.  在 `train`, `valid`, `test` 的每一个文件夹内，再创建所有类别的子文件夹 (如 `AnnualCrop`, `Forest` 等)。
4.  根据 CSV 文件的内容，将 `EuroSAT_RGB` 中的图片复制到对应的目标位置。

**最终目录结构**: 
```
- EuroSAT_Split_Physical/
  |- train/
  |  |- AnnualCrop/
  |  |  |- AnnualCrop_1.jpg
  |  |- Forest/
  |  |- ...
  |- valid/
  |  |- ...
  |- test/
     |- ...
```

**操作指南**: 
1.  **确保您已经运行过 `data_preprocessing.ipynb`** 并且 `split_info` 文件夹及其中的CSV文件已经存在。
2.  **确认下方单元格中的路径**设置正确。
3.  从上到下依次运行所有单元格。

In [ ]:
# 导入必要的库
import os
import pandas as pd
import shutil  # 用于文件复制
from tqdm import tqdm # 用于显示进度条，需要 pip install tqdm

## 1. 配置路径

请确认以下路径设置与您的项目结构一致。

In [ ]:
# --- 关键配置区 ---

# 包含 EuroSAT_RGB 和 split_info 文件夹的父目录
project_path = "F:/vscode project/IMA_PW3/EuroSat_classification-main_data_wx/"

# 原始图片文件夹路径
source_image_path = os.path.join(project_path, "EuroSAT_RGB/").replace('\\', '/')

# 包含 CSV 文件的元数据文件夹路径
metadata_path = os.path.join(project_path, "split_info/").replace('\\', '/')

# 将要创建的新文件夹，用于存放物理划分后的数据
destination_path = os.path.join(project_path, "EuroSAT_Split_Physical/").replace('\\', '/')

print(f"原始图片来源: {source_image_path}")
print(f"元数据 (CSV) 来源: {metadata_path}")
print(f"目标输出目录: {destination_path}")

## 2. 定义文件复制函数

In [ ]:
def copy_files_from_csv(csv_path, source_base, dest_base, split_name):
    """
    读取CSV文件，并将对应的图片从源目录复制到目标目录。
    
    参数:
        csv_path (str): CSV文件的路径 (例如 '.../train.csv')。
        source_base (str): 原始图片文件夹的根路径 (EuroSAT_RGB)。
        dest_base (str): 物理划分后新文件夹的根路径 (EuroSAT_Split_Physical)。
        split_name (str): 数据集名称 ('train', 'valid', 'test')。
    """
    if not os.path.exists(csv_path):
        print(f"错误: CSV文件 '{csv_path}' 未找到。请先运行数据预处理脚本。")
        return

    df = pd.read_csv(csv_path)
    
    print(f"\n--- 正在复制 '{split_name}' 数据集 ({len(df)} 张图片) ---")

    # 使用tqdm创建进度条
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        relative_path = row['ImagePath']
        
        # 构建源文件和目标文件的完整路径
        source_file = os.path.join(source_base, relative_path).replace('/', os.sep)
        # 目标路径包含 split_name, 例如 '.../train/AnnualCrop/AnnualCrop_1.jpg'
        dest_file = os.path.join(dest_base, split_name, relative_path).replace('/', os.sep)
        
        # 创建目标文件夹 (如果不存在的话)
        dest_folder = os.path.dirname(dest_file)
        os.makedirs(dest_folder, exist_ok=True)
        
        # 复制文件
        shutil.copy2(source_file, dest_file)

    print(f"'{split_name}' 数据集复制完成。")

# --- 主流程 ---

# 首先，检查并创建总的目标文件夹
print(f"创建主输出目录: {destination_path}")
os.makedirs(destination_path, exist_ok=True)

# 定义要处理的数据集
splits = ['train', 'valid', 'test']

# 循环处理每个数据集
for split in splits:
    csv_file_path = os.path.join(metadata_path, f'{split}.csv')
    copy_files_from_csv(csv_file_path, source_image_path, destination_path, split)

print("\n--- 所有文件已成功复制并划分到物理文件夹中! ---")

## 3. (可选) 如何在主代码中使用新的物理文件夹

如果您想让您的主训练代码 `EuroSat_ResNeXt_Classifier.ipynb` 使用这些新创建的物理文件夹，您需要做一些修改。

**A. 使用 PyTorch 内置的 `ImageFolder`**

这种方法更简单，不再需要我们自定义的 `EuroSAT.py` 文件。您可以在主 Notebook 中直接定义数据集：

```python
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# 新的数据根目录
physical_data_path = "F:/vscode project/IMA_PW3/EuroSat_classification-main_data_wx/EuroSAT_Split_Physical/"

# 定义图像转换
transformToTensor = transforms.Compose([
    transforms.Resize((64,)),
    transforms.CenterCrop((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 使用 ImageFolder 加载数据
train_dataset = ImageFolder(root=os.path.join(physical_data_path, 'train'), transform=transformToTensor)
valid_dataset = ImageFolder(root=os.path.join(physical_data_path, 'valid'), transform=transformToTensor)
test_dataset = ImageFolder(root=os.path.join(physical_data_path, 'test'), transform=transformToTensor)

# DataLoader 的创建方式保持不变
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
```

**B. 修改 `EurosatDataset` 类**

如果您仍想使用自定义的 `EurosatDataset` 类，您需要将其逻辑简化，让它直接从 `train`, `valid`, `test` 文件夹中读取数据，类似于 `ImageFolder` 的工作方式。但通常情况下，直接使用 `ImageFolder` 会更方便。